# 🏋️ 병렬 처리 실습 — CPU-bound·I/O-bound와 GIL

📖 **자료 연계**: 「멀티프로세싱/멀티스레딩 기반 병렬 처리 패턴」

원본 교안의 6·7교시 실습(로그 100개·1.47GB, Docker 불필요)을 더 작게 줄이고,
표준 라이브러리(`concurrent.futures`)만으로 4교시의 **가설 1·2·3**을 직접
재보는 노트북입니다. Numpy 벡터화도 "빠르게 만드는 또 다른 방법"으로
같이 비교합니다.

**✅ 완료 기준**
- [ ] CPU-bound 작업에 스레드를 붙여도 빨라지지 않는 것을 직접 봤다 (가설 1)
- [ ] 같은 작업에 프로세스를 붙이면 빨라지는 것을 확인했다
- [ ] I/O-bound 작업(대기)에는 스레드가 크게 도움이 되는 것을 확인했다 (가설 2)
- [ ] 큰 데이터를 통째로 넘길 때와 경로/크기만 넘길 때의 비용 차이를 확인했다 (가설 3)
- [ ] 🔰 미션: 합성 로그 파일을 만들어 순차 vs 병렬 파싱 속도를 직접 비교했다

> ⚠️ **시작 전 확인**: 표준 라이브러리(`concurrent.futures`, `multiprocessing`)만
> 있으면 됩니다. `psutil`이 있으면 코어별 사용률도 볼 수 있지만 없어도 진행에
> 지장 없습니다. 이 환경은 코어 2개 기준입니다 — 절대 배수(예: "8배") 대신
> **"몇 배"인지 직접 나온 숫자로 판단**하세요.

In [1]:
# ─── 환경 확인 ────────────────────────────────────────────────────────
import os
import time
import math
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

print(f"코어 수(os.cpu_count()): {os.cpu_count()}")

try:
    import psutil
    print(f"✅ psutil {psutil.__version__} — 코어별 사용률 확인 가능")
    HAS_PSUTIL = True
except ImportError:
    print("ℹ️ psutil 없음 — 코어별 사용률 셀은 건너뜁니다 (나머지는 정상 진행)")
    HAS_PSUTIL = False

코어 수(os.cpu_count()): 22
✅ psutil 7.2.2 — 코어별 사용률 확인 가능


In [2]:
import sys

a = [1, 2, 3]
b = a                          # ← 참조가 하나 늘어남 (카운트 2)
print(sys.getrefcount(a))      # 예상 출력: 3 (getrefcount 인자로 넘긴 것도 1개로 셈)
del b                          # ← 참조가 하나 줄어듦 (카운트 1)
print(sys.getrefcount(a))      # 예상 출력: 2

3
2


---
## 1부 · CPU-bound 작업 — 스레드를 늘려도 빨라지지 않는다 (가설 1)

📖 **자료 연계**: 2교시·4교시·5교시 실험 1

| 셀 | 단계 | 목표 |
|----|------|------|
| Step 1-① | 그대로 실행 | 순차 처리 기준 시간 측정 (코어별 사용률 포함) |
| Step 1-② | 그대로 실행 | ThreadPoolExecutor로 같은 작업 — 가설 1 검증 |
| Step 1-③ | 그대로 실행 | ProcessPoolExecutor로 같은 작업 — 진짜 병렬화 |
| Step 1-④ | 한 곳만 바꾸기 | Numpy 벡터화로 같은 계산을 다시 — 셋을 나란히 비교 |

In [3]:
# ─── Step 1-① 그대로 실행 — 순차 처리 기준 (CPU-bound) ─────────────────
# 📖 자료 연계: 「2026-09-14 ...」 2교시 모듈 2-1 · 4교시 모듈 4-2
#
# 작업을 "숫자 20만 개를 8묶음으로 나눈 덩어리"로 던집니다. 숫자 하나하나를
# 낱개로 던지면(태스크 20만 개) 풀에 일감을 나눠주는 오버헤드 자체가 계산
# 시간보다 커져서 비교가 무의미해집니다 — 3교시 "컨텍스트 스위칭 비용"과
# 같은 이유로, 태스크는 적당히 굵게 나누는 것이 정석입니다.

def is_prime(n: int) -> bool:
    # 소수 판별 — 순수 파이썬 계산, CPU-bound의 대표 예시.
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True


def check_chunk(numbers):
    # 숫자 묶음 하나를 통째로 받아 각각 판별한다 — 이것이 "태스크 1개"의 단위.
    return [is_prime(n) for n in numbers]


NUMBERS = list(range(10_000_000, 10_200_000))  # 20만 개
N_CHUNKS = 8
_size = len(NUMBERS) // N_CHUNKS
CHUNKS = [NUMBERS[i * _size:(i + 1) * _size] for i in range(N_CHUNKS)]

if HAS_PSUTIL:
    import psutil
    psutil.cpu_percent(interval=None, percpu=True)  # 첫 호출은 기준점이라 버림

t0 = time.perf_counter()
result_seq = [is_prime(n) for n in NUMBERS]
t1 = time.perf_counter()
seq_time = t1 - t0

if HAS_PSUTIL:
    usage = psutil.cpu_percent(interval=None, percpu=True)
    print(f"코어별 사용률(참고, 순간값): {usage}")

print(f"순차 처리 시간: {seq_time:.3f}s (소수 {sum(result_seq)}개 발견)")

코어별 사용률(참고, 순간값): [4.2, 0.0, 0.0, 0.0, 0.0, 1.2, 0.0, 0.0, 1.2, 0.0, 55.4, 0.0, 0.0, 42.2, 0.0, 42.9, 22.9, 0.0, 12.5, 0.0, 0.0, 0.0]
순차 처리 시간: 1.298s (소수 12391개 발견)


In [4]:
# ─── Step 1-② 그대로 실행 — ThreadPoolExecutor (가설 1 검증) ───────────
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    result_thread = [x for chunk in ex.map(check_chunk, CHUNKS) for x in chunk]
t1 = time.perf_counter()
thread_time = t1 - t0

print(f"스레드(4개) 처리 시간: {thread_time:.3f}s")
print(f"순차 대비 배수: {seq_time / thread_time:.2f}배  ← 1.0에 가까울수록(또는 그 이하) 가설 1이 맞다는 뜻")
print("결과 일치:", result_seq == result_thread)

# 💡 핵심: CPU-bound 작업은 GIL 때문에 스레드를 늘려도 "동시에 계산"되지 않습니다.
#          오히려 스레드 전환(컨텍스트 스위칭) 비용만 붙어 더 느려질 수 있습니다.

스레드(4개) 처리 시간: 1.289s
순차 대비 배수: 1.01배  ← 1.0에 가까울수록(또는 그 이하) 가설 1이 맞다는 뜻
결과 일치: True


In [ ]:
from concurrent.futures import ProcessPoolExecutor

def test_func(x):
    return x * x

with ProcessPoolExecutor(max_workers=2) as ex:
    result = list(ex.map(test_func, [1, 2, 3, 4]))      #파이썬 버전이랑 함수가 호환이 잘 안되어서 로컬에서는 에러가 뜨는 거 같음

print(result)

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [9]:
# ─── Step 1-③ 그대로 실행 — ProcessPoolExecutor (진짜 병렬화) ──────────
if __name__ == "__main__":
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=2) as ex:
        result_proc = [x for chunk in ex.map(check_chunk, CHUNKS) for x in chunk]
    t1 = time.perf_counter()
    proc_time = t1 - t0

    print(f"프로세스({os.cpu_count()}개) 처리 시간: {proc_time:.3f}s")
    print(f"순차 대비 배수: {seq_time / proc_time:.2f}배  ← 코어 수에 가까울수록 이상적")
    print("결과 일치:", result_seq == result_proc)

# ⚠️ 흔한 실수: check_chunk처럼 모듈 최상위에 정의된 함수만 ProcessPoolExecutor에 넘길 수 있습니다.
#              람다나 노트북 셀 안의 지역 함수는 pickle이 안 되어 에러가 납니다.

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [ ]:
# ─── Step 1-④ 한 곳만 바꾸기 — Numpy 벡터화는 어떨까? ───────────────────
# TODO(🔰): NUMBERS 범위를 바꿔서 재실행해보고 배수가 어떻게 달라지는지 보세요.
import numpy as np

def is_prime_vectorized(arr: np.ndarray) -> np.ndarray:
    # 배열 전체에 대해 소수 여부를 벡터화 연산으로 판별한다.
    arr = np.asarray(arr)
    is_p = arr >= 2
    max_check = int(arr.max() ** 0.5) + 1
    for i in range(2, max_check):
        is_p &= (arr % i != 0) | (arr == i)
    return is_p

t0 = time.perf_counter()
result_vec = is_prime_vectorized(np.array(NUMBERS))
t1 = time.perf_counter()
vec_time = t1 - t0

print(f"Numpy 벡터화 시간: {vec_time:.3f}s")
print(f"순차 대비 배수: {seq_time / vec_time:.2f}배")
print("결과 일치:", list(result_vec) == result_seq)

print()
print("=== 세 방식 요약 ===")
print(f"순차        : {seq_time:.3f}s (기준)")
print(f"스레드      : {thread_time:.3f}s ({seq_time/thread_time:.2f}배)")
print(f"Numpy 벡터화: {vec_time:.3f}s ({seq_time/vec_time:.2f}배)")

# 💡 핵심: 이 문제에서는 벡터화가 오히려 더 느리게 나올 수 있습니다 — 이상한 게 아닙니다.
#          순차 is_prime()은 나누어떨어지는 순간 바로 return(조기 종료)하지만,
#          벡터화 버전은 배열 전체에 "같은 연산을 똑같이" 적용해야 하므로 이미 답이
#          나온 숫자도 나머지 후보 i를 계속 검사합니다. 소수 판별처럼 "값마다 분기·조기
#          종료가 중요한" 문제는 벡터화가 잘 안 맞는 대표 사례입니다. 반대로 (Numpy
#          노트북에서 봤던) 배열 전체에 동일한 연산 하나만 적용하는 문제라면 벡터화가
#          압도적으로 빠릅니다. "무조건 빠른 방법"은 없고, 문제 성격에 맞는 방법을
#          고르는 것이 핵심입니다. 멀티프로세싱은 "일손을 늘리는 것", 벡터화는
#          "반복을 C에 넘기는 것"— 서로 다른 방식으로 빨라지므로 함께 쓸 수도 있습니다.

---
## 2부 · I/O-bound 작업 — 스레드가 크게 도움이 된다 (가설 2)

📖 **자료 연계**: 4교시 모듈 4-2 · 5교시 실험 2

In [ ]:
# ─── Step 2-① 그대로 실행 — I/O 대기를 흉내낸 작업 ─────────────────────
# 실제 네트워크 요청 대신 time.sleep()으로 "기다리는 작업"을 흉내냅니다.
# time.sleep()은 GIL을 반납하므로 실제 I/O 대기와 동일한 성질을 보입니다.

def wait_like_io(_: int) -> int:
    time.sleep(0.05)  # 50ms 대기 — HTTP 요청 하나를 흉내
    return 1

TASKS = list(range(40))  # 40번 "요청"

t0 = time.perf_counter()
seq_io = [wait_like_io(t) for t in TASKS]
t1 = time.perf_counter()
seq_io_time = t1 - t0
print(f"순차 처리(40회 × 50ms 대기): {seq_io_time:.3f}s")

In [ ]:
# ─── Step 2-② 그대로 실행 — ThreadPoolExecutor로 대기를 겹치기 ─────────
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as ex:
    thread_io = list(ex.map(wait_like_io, TASKS))
t1 = time.perf_counter()
thread_io_time = t1 - t0

print(f"스레드(8개) 처리: {thread_io_time:.3f}s")
print(f"순차 대비 배수: {seq_io_time / thread_io_time:.1f}배  ← 1부의 배수와 비교해보세요")

# 💡 핵심: 이번엔 스레드가 크게 이깁니다. time.sleep() 동안 GIL이 풀려 있어서
#          다른 스레드가 그 틈에 자기 대기를 시작할 수 있기 때문입니다 — "대기를 겹친다"는 게 이런 뜻입니다.
# ⚠️ 흔한 실수: 1부(CPU-bound)와 2부(I/O-bound)에 같은 도구(스레드)를 썼는데 결과가 정반대입니다.
#              "스레드가 좋다/나쁘다"가 아니라 "무엇을 기다리느냐"가 기준입니다.

---
## 3부 · 직렬화 비용 — 무엇을 넘기느냐가 속도를 가른다 (가설 3)

📖 **자료 연계**: 4교시 모듈 4-3

In [ ]:
# ─── Step 3-① 그대로 실행 — 큰 데이터를 직접 넘기기 vs 크기만 넘기기 ───
# 📓 두 워커 함수 모두 모듈 최상위에 정의해야 pickle이 됩니다.

def sum_only(data):
    # 이미 만들어진 큰 리스트를 그대로 받아서 더한다 — 리스트 자체를 pickle로 전송.
    return sum(data)


def build_and_sum(n):
    # 정수 하나만 받고, 워커 안에서 직접 리스트를 만든다 — 전송량이 거의 0.
    return sum(range(n))


N = 3_000_000
WORKERS = 4
big_lists = [list(range(N)) for _ in range(WORKERS)]  # 미리 만들어 둔 "큰 데이터"

if __name__ == "__main__":
    # ❌ 큰 리스트를 통째로 넘김
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=WORKERS) as ex:
        r1 = list(ex.map(sum_only, big_lists))
    t1 = time.perf_counter()
    big_transfer_time = t1 - t0

    # ✅ 크기(정수 하나)만 넘기고 워커가 직접 만듦
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=WORKERS) as ex:
        r2 = list(ex.map(build_and_sum, [N] * WORKERS))
    t1 = time.perf_counter()
    small_transfer_time = t1 - t0

    print(f"❌ 큰 리스트 통째로 전송   : {big_transfer_time:.3f}s")
    print(f"✅ 크기만 전송(워커가 생성): {small_transfer_time:.3f}s")
    print(f"차이: {big_transfer_time / small_transfer_time:.2f}배")
    print("결과 일치:", r1 == r2)

# 💡 핵심: 로그 병렬 파싱에서 "파일 내용을 직접 넘기지 않고 파일 경로만 넘기는" 이유가 이것입니다.
#          워커가 자기 몫을 스스로 만들거나 읽게 하면, 부모↔자식 사이에 오가는 데이터가 작아집니다.

---
## 🔰 종합 미션 — 합성 로그 파일 병렬 파싱

📖 **자료 연계**: 6교시 필수 실습 (원본은 로그 100개·1.47GB, 여기서는 축소판)

In [ ]:
# ─── 🔰 미션 1/2 — 합성 로그 파일 여러 개 만들기 ────────────────────────
# Numpy로 값들을 한 번에 뽑아낸 뒤 문자열로 조립합니다 — 파이썬 random을
# 줄마다 호출하는 것보다 훨씬 빠릅니다(1부에서 배운 벡터화의 실전 활용).
import os

os.makedirs("data/logs", exist_ok=True)

LEVELS = np.array(["INFO", "WARN", "ERROR"])
LEVEL_P = [0.727, 0.182, 0.091]
N_FILES = 8
LINES_PER_FILE = 300_000

def make_log_file(path, n_lines, seed):
    rng = np.random.default_rng(seed)
    hours = rng.integers(0, 24, size=n_lines)
    levels = rng.choice(LEVELS, size=n_lines, p=LEVEL_P)
    comps = rng.integers(1, 6, size=n_lines)
    lines = [
        f"2026-09-14 {h:02d}:00:00 {lv} component-{c}: message {i}"
        for i, (h, lv, c) in enumerate(zip(hours, levels, comps))
    ]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

for i in range(N_FILES):
    make_log_file(f"data/logs/log_{i:02d}.log", LINES_PER_FILE, seed=1000 + i)

print(f"data/logs/ 에 로그 파일 {N_FILES}개 생성 완료 (파일당 {LINES_PER_FILE:,}줄, 총 {N_FILES*LINES_PER_FILE:,}줄)")

In [ ]:
# ─── 🔰 미션 2/2 — 순차 vs 병렬 파싱 속도 비교 ──────────────────────────
# TODO(🔰): N_FILES나 LINES_PER_FILE을 늘려보고 배수가 어떻게 바뀌는지 확인하세요.
from pathlib import Path
from collections import defaultdict

def parse_file(path):
    # 파일 하나를 읽어 시간대별 ERROR 건수를 센다 — 경로만 받아 워커가 직접 읽음(가설 3 적용).
    counts = defaultdict(int)
    with open(path, encoding="utf-8") as f:
        for line in f:
            if " ERROR " in line:
                hour = line[11:13]  # "HH" 부분만 슬라이싱
                counts[f"hour:{hour}"] += 1
    return dict(counts)


paths = [str(p) for p in sorted(Path("data/logs").glob("*.log"))]

# ❌ 순차
t0 = time.perf_counter()
seq_results = [parse_file(p) for p in paths]
t1 = time.perf_counter()
seq_parse_time = t1 - t0

if __name__ == "__main__":
    # ✅ 병렬 (경로만 전송)
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
        par_results = list(ex.map(parse_file, paths))
    t1 = time.perf_counter()
    par_parse_time = t1 - t0

    # 병합 — 같은 시간대가 여러 파일에 걸쳐 나올 수 있으므로 재집계
    merged = defaultdict(int)
    for r in par_results:
        for k, v in r.items():
            merged[k] += v
    seq_merged = defaultdict(int)
    for r in seq_results:
        for k, v in r.items():
            seq_merged[k] += v

    print(f"순차 파싱: {seq_parse_time:.3f}s")
    print(f"병렬 파싱: {par_parse_time:.3f}s ({seq_parse_time / par_parse_time:.2f}배)")
    print("전체 ERROR 건수 일치:", dict(merged) == dict(seq_merged))
    print(f"총 ERROR 건수: {sum(merged.values()):,}건")

---
## 📬 자가 체크

- [ ] CPU-bound 작업에 스레드를 붙여도 빨라지지 않는 것을 직접 봤다 (가설 1)
- [ ] 같은 작업에 프로세스를 붙이면 빨라지는 것을 확인했다
- [ ] I/O-bound 작업(대기)에는 스레드가 크게 도움이 되는 것을 확인했다 (가설 2)
- [ ] 큰 데이터를 통째로 넘길 때와 경로/크기만 넘길 때의 비용 차이를 확인했다 (가설 3)
- [ ] 🔰 미션: 합성 로그 파일을 만들어 순차 vs 병렬 파싱 속도를 직접 비교했다

---
### ➡️ 다음 연결

오늘 본 것은 **"일을 나누는 법"** 이었습니다. 다음(Polars/Dask)은 같은 문제를
다른 각도에서 봅니다 — **"데이터 자체를 나누는 법"** 입니다. 둘 다 "한 코어로는
못 버티는 규모"를 다루지만, 오늘은 작업(태스크) 단위로, 다음은 데이터(행·열)
단위로 나눈다는 차이가 있습니다.